# 2-clean&filter

## 2.1 pré-nettoyage, pré-filtrage et pré-recodages

In [1]:
import pandas as pd
import re

# Charger le df concaténé des deux législatures
df = pd.read_csv(
    "../data/interim/extract_15_16_concat.csv",
    low_memory=False,
    dtype={
        "id_orateur": str  # éviter identification en float avant d'avoir ajouté le "PA"
    },
)

print("Shape du df chargé : ", df.shape)

# ==============================
# Pré-nettoyage et pré-filtrage
# ==============================
"""
nb : précision choix 
- exclusion président.e :
role_debat n'est pas bien identifié, utiliser nom_orateur
(avant de le recoder/nettoyer car sinon risque perte par remplacement)
- id_acteur vs id_orateur :
certains cas (~3000) id_orateur plus précis (un code PA) que id_acteur qui a PA0
Mais en fait ce sont des 100% interruptions avec quasi toujours plusieurs locuteurs.
id_orateur en renvoie (mal) un seul -> on préfère garder le PA0 (neutre)
TODO : Rares exceptions avec Dupont-morretti seul, etc. -> mais bon…
"""

# Exclure les prises de parole de "Mme la présidente" et "M. le président"
df = df[~df["nom_orateur"].str.strip().isin(["M. le président", "Mme la présidente"])]

# Ne garder que le code style NORMAL
df = df[df["code_style"] == "NORMAL"]

# Changer les missing values pour non_précisé (majoritaire) dans Code_parole
df["code_parole"] = df["code_parole"].fillna("non_précisé")

# Garder une trace de la longueur des interventions brutes
df["len_texte_brut"] = df["texte"].str.len()

# TODO: vérifier si tout OK pour fusion id_acteur/id_orateur/nom_orateur. notamment si ils correspondent bien au meme point
# TODO: vérif comparaison de id acteur =! id orateur

# Stabiliser le id_orateur pour être au format AN
df["id_orateur"] = "PA" + df["id_orateur"]
# Remplacer les valeurs manquantes de id_acteur par id_orateur quand disponible
df["id_acteur"] = df["id_acteur"].combine_first(df["id_orateur"])

# ===========================================================
# Récupérer et nettoyer les noms les plus fréquents
# pour chaque id_acteur sauf PA0 et les id_acteur manquants
# ===========================================================

# ========== Recoder par noms les plus fréquents ==========

# Nom le plus fréquent
most_frequent_name = df.groupby("id_acteur")["nom_orateur"].agg(
    lambda x: x.dropna().mode().iloc[0] if x.dropna().size > 0 else None
)  # version plus stable que value_counts().idxmax() en cas d'ex-aequo


# Renvoyer le nom le plus fréquent sauf si id_acteur == PA0 ou id_acteur est manquant
# Limite de la fonction : invisibilise les rares cas d'interventions
# mal identifiées par leur PA, mais qui ont le bon nom
# (ici le nom majoritaire sera renvoyé)
def get_most_frequent_name(row):
    """
    Récupération de la forme la plus fréquente du nom,
    uniquement pour acteurs différents de PA0.
    if PA0 : nom brut, else : nom le plus fréquent pour cet id.
    /!\ Limite de la fonction : invisibilise les rares cas d'interventions
    mal identifiées par leur PA, mais qui ont le bon nom (ici le nom majoritaire sera renvoyé)
    """
    if row["id_acteur"] == "PA0" or pd.isna(row["id_acteur"]):
        return row["nom_orateur"]
    return most_frequent_name.get(row["id_acteur"], row["nom_orateur"])


df["nom_orateur_clean"] = df.apply(get_most_frequent_name, axis=1)


# ========== Nettoyer les noms d'orateurs ==========


def nettoyer_nom(texte):
    if not isinstance(texte, str):
        return texte
    # Supprimer les balises HTML/XML
    texte = re.sub(r"<[^>]+>", "", texte)
    # Supprimer contenu entre parenthèses
    texte = re.sub(r"\([^)]*\)", "", texte)
    # Supprimer les virgules
    texte = texte.replace(",", " ")
    # Supprimer les espaces multiples
    texte = re.sub(r"\s+", " ", texte).strip()
    # uniformise pour les apostrophes
    texte = texte.replace("’", "'")
    return texte


df["nom_orateur_clean"] = df["nom_orateur_clean"].apply(nettoyer_nom)

print("Shape du df après pré-nettoyage et pré-filtrage : ", df.shape)


Shape du df chargé :  (1128128, 29)
Shape du df après pré-nettoyage et pré-filtrage :  (683680, 31)


#### CAS CONFLITS NOMS ET OU ID_ACTEUR VS ID_ORATEUR

In [2]:
# TODO: MATTHIAS
# suite ajout comparaison nom origine vs nom clean = permet repérer erreurs id_acteur (cf cas ministre, etc.)
# DÉSORMAIS CHOISIR SI TU VEUX EN FAIRE UN TRUC OU SI TROP MARGINAL

# dessous :
# 1/ fuzzyfuzz
# 2/ aussi une version avec id_acteur vs id_orateur pour cas évidents de possible pb

In [3]:
# ========== Comparaison nom_orateur brut vs nom_orateur_clean ==========
# Repérer les cas où le nom le plus fréquent assigné à un id_acteur
# diffère du nom brut de l'intervention -> signe possible d'une erreur d'id_acteur
# (ex : ministre ou invité avec un PA qui appartient à un autre)
# Passage par un rapidfuzz pour avoir un score de similarité

from rapidfuzz import fuzz


def normaliser_nom_fuzzy(x):
    if not isinstance(x, str):
        return x
    x = nettoyer_nom(x).lower().strip()
    # retire ponctuation pour éviter de flaguer juste une virgule/point
    x = re.sub(r"[^\w\s'-]", " ", x)
    x = re.sub(r"\s+", " ", x).strip()
    return x


# Noms normalisés
# pour nom orateur = appliquer aussi le nettoyage standard pour comparabilité
nom_brut_norm = df["nom_orateur"].apply(nettoyer_nom).apply(normaliser_nom_fuzzy)
nom_clean_norm = df["nom_orateur_clean"].apply(normaliser_nom_fuzzy)

# Score fuzzy (0-100)
score_fuzzy = [
    fuzz.token_sort_ratio(a, b) if isinstance(a, str) and isinstance(b, str) else None
    for a, b in zip(nom_brut_norm, nom_clean_norm)
]
df["score_nom_fuzzy"] = score_fuzzy

# Seuil: plus haut = plus strict
seuil_similarite = 96

mask_nom_diff_significatif = (
    (df["id_acteur"] != "PA0")
    & nom_brut_norm.notna()
    & nom_clean_norm.notna()
    & (nom_brut_norm != nom_clean_norm)
    & (df["score_nom_fuzzy"] < seuil_similarite)
)

print(
    "Interventions avec différence significative nom brut vs nom clean :",
    int(mask_nom_diff_significatif.sum()),
)
print(
    "id_acteur uniques concernés :",
    int(df.loc[mask_nom_diff_significatif, "id_acteur"].nunique()),
)

fuzz_pb = df.loc[
    mask_nom_diff_significatif,
    ["id_acteur", "id_orateur", "nom_orateur", "nom_orateur_clean", "score_nom_fuzzy"],
].sort_values("score_nom_fuzzy")
display(fuzz_pb.head())

Interventions avec différence significative nom brut vs nom clean : 2889
id_acteur uniques concernés : 147


,id_acteur,id_orateur,nom_orateur,nom_orateur_clean,score_nom_fuzzy
927443,PA1008,PA1008,Un député du groupe RN,M. Alain David,17.142857
332608,PA605991,PA605991,Plusieurs députés,Mme Annie Genevard,17.142857
808306,PA607619,PA267306,M. André Chassaigne,M. Paul Molac,20.000000
29761,PA721824,PA721824,M. Éric Poulliat,M. Hugues Renson,20.000000
1070377,PA717379,PA717379,Un député du groupe RE,M. Sylvain Maillard,20.000000


In [4]:
# NB : en gros au dessus de ~60 c'est plus que :
# - des doubles intervenants
# - des prénoms différents
# - et trucs mineurs)

deduplication_pb_fuzz = (
    fuzz_pb[
        [
            "id_acteur",
            "id_orateur",
            "nom_orateur",
            "nom_orateur_clean",
            "score_nom_fuzzy",
        ]
    ]
    .value_counts()
    .reset_index(name="n")
    .sort_values("score_nom_fuzzy")
)
deduplication_pb_fuzz

,id_acteur,id_orateur,nom_orateur,nom_orateur_clean,score_nom_fuzzy,n
128,PA605991,PA605991,Plusieurs députés,Mme Annie Genevard,17.142857,1
153,PA1008,PA1008,Un député du groupe RN,M. Alain David,17.142857,1
137,PA607619,PA267306,M. André Chassaigne,M. Paul Molac,20.000000,1
171,PA717379,PA717379,Un député du groupe RE,M. Sylvain Maillard,20.000000,1
56,PA721824,PA721824,M. Éric Poulliat,M. Hugues Renson,20.000000,1
...,...,...,...,...,...,...
189,PA720764,PA720764,Mme Florence Lasserre (Dem),Mme Florence Lasserre-David,87.500000,1
9,PA718728,PA718728,Mme Laurence Vanceunebrock-Mialon,Mme Laurence Vanceunebrock,88.135593,34
108,PA791812,PA791812,Sophia Chikirou,Mme Sophia Chikirou,88.235294,1
121,PA-121559,PA-121559,Emmanuelle Auriol,Mme Emmanuelle Auriol,89.473684,1


In [5]:
deduplication_pb_fuzz.to_csv("../data/temp/deduplication_pb_fuzz.csv", index=False)

In [6]:
# Même chose pour les cas où id_acteur et id_orateur sont différents (mais pas PA0)
id_diff = (
    df.loc[
        df["id_acteur"].notna()
        & df["id_orateur"].notna()
        & (df["id_acteur"] != "PA0")
        & (df["id_orateur"] != "PA0")
        & (df["id_acteur"] != df["id_orateur"]),
        ["id_acteur", "id_orateur", "nom_orateur", "nom_orateur_clean"],
    ]
    .value_counts()
    .reset_index(name="n")
    .sort_values(["n", "id_acteur", "id_orateur"], ascending=[False, True, True])
)

print("Nombre de couples id_acteur / id_orateur différents :", len(id_diff))
display(id_diff)

Nombre de couples id_acteur / id_orateur différents : 46


,id_acteur,id_orateur,nom_orateur,nom_orateur_clean,n
0,PA793940,PA719472,M. Jean-René Cazeneuve,M. Thomas Cazenave,2
1,PA1012,PA609590,M. Charles de la Verpillière,M. Charles de la Verpillière,1
13,PA1206,PA266797,M. Nicolas Dupont-Aignan,M. Nicolas Dupont-Aignan,1
26,PA1567,PA795636,M. Benjamin Lucas,M. Jérôme Guedj,1
27,PA2150,PA719676,M. Jean-Luc Mélenchon,M. Jean-Luc Mélenchon,1
28,PA266788,PA267042,M. Pierre Morel-À-L’Huissier,M. Pierre Morel-À-L'Huissier,1
29,PA267042,PA331582,M. Yannick Favennec Becot,M. Yannick Favennec Becot,1
30,PA267585,PA721486,M. Francis Vercamer,M. Francis Vercamer,1
31,PA331582,PA721976,M. Philippe Vigier,M. Philippe Vigier,1
32,PA332228,PA266797,M. Thierry Benoit,M. Thierry Benoit,1


In [7]:
id_diff.to_csv("../data/temp/deduplication_pb_id_diff.csv", index=False)

## 2.2 Match des infos sur les députés (données datan)

### 2.2.1 Match des infos générales

In [8]:
# ==============================
# MATCH DONNÉES DÉPUTÉS
# ==============================

df_deputes = pd.read_csv("../data/raw/id-dep/deputes-historique(datan-datagouv).csv")

# suppression des colonnes non utiles qui introduisent soucis parsing
df_deputes = df_deputes.drop(
    columns=[
        "mail",
        "twitter",
        "facebook",
        "website",
        "active",
        "scoreParticipationSpecialite",
        "datePriseFonction",
        "groupe",
        "naissance",
    ]
)

# ======Fusion des données députés======

print("shape avant fusion:", df.shape)

assert df_deputes["id"].is_unique, "ids du df_deputes non uniques !"

# Merger et virer la col id pour éviter doublon
df = df.merge(
    df_deputes,
    left_on="id_acteur",
    right_on="id",
    how="left",
    suffixes=("", "_dep"),
    validate="many_to_one",  # check if merge keys are unique in right dataset
).drop(columns=["id"])  # supprimer la colonne id du df_deputes

print("shape après fusion données députés:", df.shape)

shape avant fusion: (683680, 32)
shape après fusion données députés: (683680, 49)


### 2.2.2 Match temporel des affiliations

In [9]:
# ======================================================
# AFFILIATION PARTISANE
# Logique suivie :
# 1. récupérer le groupe a date d'intervention si dispo
# 2. fallback sur dernière affiliation connue (groupe/groupeabrev)
# 3. variable supplémentaire avec gouvernement
# = écraser affil par GOUV si qualite_orateur précise fonction gouvernementale
# ======================================================


# ======================================================
# RECODAGE ET MATCH TEMPOREL DES AFFILIATIONS PARTISANES
# cf. affiliation lors de telle prise de parole
# ======================================================


# ========== Recodage des dénominations de groupes ==========
"""
nb : ici choix de recoder avec les nom des partis,
car ils sont moins sensible aux évolutions marginales de noms,
même si en réalité les groupes parlementaires sont + larges que les partis
et peuvent servir à accueillir des NI d'étiquettes diverses
"""

# Lecture du fichier d'affiliation par périodes
df_affiliation = pd.read_csv(
    "../data/raw/id-dep/datan_affiliations.csv", encoding="latin1", sep=";"
)  # format dégueu

# Recodage des partis pour stabilité temporelle des noms
recodage_affiliation = {
    "RE": "REN",
    "EPR": "REN",
    "LAREM": "REN",
    "MODEM": "DEM",
    "SOC": "SOC-A",
    "NG": "SOC-A",
    "LFI-NUPES": "LFI",
    "FI": "LFI",
    "UDI-AGIR": "UDI",
    "UDI-A-I": "UDI",
    "LC": "UDI",
    "UDI_I": "UDI",
    "UDI-I": "UDI",
    "ECOLO": "ECO",
    "GDR-NUPES": "GDR",
    "LT": "LIOT",
    # Garde pour trace mais pas nécessaire car pas de changement
    # "LIOT": "LIOT",
    # "LR": "LR",
    # "RN": "RN",
    # "MODEM": "MODEM",
    # "LFI": "LFI",
    # "HOR": "HOR",
    # "DEM": "DEM",
}

# Application du recodage des noms de partis au df d'affiliation
df_affiliation["libelleAbrev"] = df_affiliation["libelleAbrev"].astype(str).str.strip()
df_affiliation["parti_recod"] = df_affiliation["libelleAbrev"].replace(
    recodage_affiliation
)

# ========== Match temporel des affiliations ==========

"""
nb : Plutôt qu'un merge foireux, parti sur un lookup ligne à ligne
(= pb des orateurs non députés qui étaient pas présents, etc.)
Le fichier est suffisamment réduit pour que le surplus de calcul soit pas un pb
nb : attention aux bornes temporelles (cf.normalize() pour ignorer l'heure)
"""

# préparation des dates
df["dateSeance_ts"] = pd.to_datetime(
    df["dateSeance"], format="%Y%m%d%H%M%S%f", errors="raise"
)
df_affiliation["dateDebut"] = pd.to_datetime(
    df_affiliation["dateDebut"], errors="raise"
)
df_affiliation["dateFin"] = pd.to_datetime(df_affiliation["dateFin"], errors="raise")
# aviser si jamais besoin un jour de traiter des affiliations en cours
# df_affiliation["dateFin"] = df_affiliation["dateFin"].fillna(pd.Timestamp("2100-01-01"))

# indexer par mpId pour lookup rapide
aff_by_mp = {
    mp: g[["dateDebut", "dateFin", "parti_recod"]].to_dict("records")
    for mp, g in df_affiliation.groupby("mpId")
}


# Fonction de recodage temporel des affiliations
def get_parti_for_row(row):
    """
    Retourne l'affiliation partisane recodée correspondant à la date de séance.

    La fonction :
    - lit `id_acteur` (assimilé à `mpId`) et `dateSeance_ts` sur la ligne ;
    - parcourt les périodes d'affiliation de ce député (si présent dans aff_by_mp);
    - renvoie `parti_recod` si `dateSeance_ts` (normalisée au jour) est comprise
    entre `dateDebut` et `dateFin` (bornes incluses).
    """
    mp = row.get("id_acteur")  # correspond au mpId
    # gérer le cas des orateurs non députés ou autre type intervention
    if pd.isna(mp) or mp not in aff_by_mp:
        return None
    # récupérer le ts de l'intervention
    ts = row.get("dateSeance_ts")
    if pd.isna(ts):
        return None
    # retourner l'affiliation qui colle à la date d'intervention
    for rec in aff_by_mp[mp]:
        # attention : .normalize() pour ignorer l'heure car sinon hors des bornes de fin
        if rec["dateDebut"] <= ts.normalize() <= rec["dateFin"]:
            return rec["parti_recod"]
    return None


# application du match temporel
df["affiliation_mandat_députés"] = df.apply(get_parti_for_row, axis=1)

# Pas parfait mais pour avoir une idée :
print(
    "affectés :",
    df["affiliation_mandat_députés"].notna().sum(),
    # Eux on sait pas (pas députés, autre code parole intervention, etc.)
    "| non affectés :",
    df["affiliation_mandat_députés"].isna().sum(),
    # nb cas unique
    "| id_acteur sans affiliation dynamique :",
    df[df["affiliation_mandat_députés"].isna()]["id_acteur"].nunique(),
)


affectés : 563127 | non affectés : 120553 | id_acteur sans affiliation dynamique : 234


### 2.2.3 Fallback des affiliations manquantes

#### Forcer le renvoi d'une affiliation si groupeAbrev connu

In [10]:
# ============================================================
# GESTION AFFILIATIONS MANQUANTES ET MEMBRES DU GOUVERNEMENT
# - Fallback pour les affiliations manquantes
# - Gestion des cas limites (RN, etc.)
# - Création catégorie "GOUV" pour les membres du gouvernement

# ============================================================

# ========= Fallback affiliation manquantes par groupeAbrev ==========

# TODO: aviser si va à Matthias + ce que veux faire de EDS / AGIR-E
# TODO: veut aussi dire qu'on bourre des cas limites, genre un vieux député UMP qui revient au gouv etc.
# Aussi possible de gérer à la main les cas UMP, etc.

# Forcer une affiliation avec le groupe "groupeAbrev" du fichier info députés
df["affiliation"] = df["affiliation_mandat_députés"].combine_first(df["groupeAbrev"])
# réutiliser le même recodage que pour les affiliations
df["affiliation"] = df["affiliation"].replace(recodage_affiliation)
# Et gérer les nouvelles dénominations propres groupeAbrev
df["affiliation"] = df["affiliation"].replace({"LES-REP": "LR", "UMP": "LR"})

# Diagnostic des cas concernés par le fallback groupeAbrev
# = c'est surtout des membres du gouv qui avaient pas d'affiliation de mandat députés
# + quelques rares cas d'interv députés ou on manque parfois l'affiliation dynamique (bornes ?)

mask_fallback = df["affiliation_mandat_députés"].isna() & df["affiliation"].notna()

print("=== Cas concernés par le fallback via groupeAbrev ===")
print("Nombre d'interventions concernées :", mask_fallback.sum())
print(
    "Nombre d'id_acteur uniques concernés :",
    df.loc[mask_fallback, "id_acteur"].nunique(dropna=True),
)

print("\nListe des orateurs concernés :")
print(df.loc[mask_fallback, "nom_orateur_clean"].dropna().unique())

# print(
#     "\nNombre restant d'interventions sans affiliation :",
#     df["affiliation"].isna().sum(),
# )
# print(
#     "Nombre restant d'id_acteur uniques sans affiliation :",
#     df[df["affiliation"].isna()]["id_acteur"].nunique(),
# )


=== Cas concernés par le fallback via groupeAbrev ===
Nombre d'interventions concernées : 64950
Nombre d'id_acteur uniques concernés : 60

Liste des orateurs concernés :
['M. Bruno Le Maire' 'M. Édouard Philippe' 'M. Olivier Dussopt'
 'M. Benjamin Griveaux' 'M. Stéphane Travert' 'Mme Brune Poirson'
 'M. Christophe Castaner' 'M. Jean-Yves Le Drian' 'Mme Bérangère Abba'
 'Mme Barbara Pompili' 'Mme Élisabeth Borne' 'M. Jean-Baptiste Djebbari'
 'Mme Nathalie Elimas' 'Mme Brigitte Bourguignon' 'Mme Brigitte Klinkert'
 'Mme Annick Girardin' 'M. Mounir Mahjoubi' 'M. Gérald Darmanin'
 'M. Olivier Véran' 'Mme Agnès Pannier-Runacher' 'M. Marc Fesneau'
 'M. Clément Beaune' 'Mme Sarah El Haïry' 'Mme Geneviève Darrieussecq'
 'M. Franck Riester' 'Mme Olivia Grégoire' 'M. Laurent Pietraszewski'
 'M. François de Rugy' 'Mme Amélie de Montchalin' 'Mme Nadia Hai'
 'Mme Roselyne Bachelot' 'Mme Christelle Dubos' 'M. Gabriel Attal'
 'M. Adrien Taquet' 'M. Joël Giraud' 'Mme Valérie Boyer'
 'Mme Prisca Theven

#### Forcer affiliation des RN qui étaient en NI (étaient pas assez pour groupe)

In [11]:
# ========= Gestion cas limites RN ==========

# Recodage des RN de la XVe législature au bloc RN
# nb = choix = initialement en NI car pas assez nombreux pour former un groupe

liste_NI_RN = [
    "PA720822",  # Bruno Bilde
    "PA720668",  # Sébastien Chenu
    "PA720468",  # Emmanuel Blairy
    "PA720614",  # Marine Le Pen
    "PA719436",  # Nicolas Meizonnet
    "PA720802",  # Catherine Pujol
    "PA719608",  # Emmanuelle Ménard, rattachée au RN entre 2017 et 2022 mais plus entre 2022 et 2024
    "PA720606",  # Ludovic Pajot
    "PA606212",  # Gilbert Collard
    "PA720798",  # Louis Aliot
    # "PA720610", # Myriane Houplain, rattachée au RN entre 2017 et 2022, part ensuite reconquête ?
    # TODO : aviser du cas de Houplain que tu citais en todo mais pas ici
    # possible choix diff de ta part car elle part ensuite reconquête ?
    # mais si on applique même choix ça pourrait coller avec le fait
    # que pendant sa période d'intervention elle était rattachée au RN mais pas assez nb pour groupe ?
    # TODO : voir si d'autres cas NI/RN ? (cf multi affil)
]

# Date seuil : fin de la 15e législature
date_seuil = pd.Timestamp("2022-06-21")

# Condition combinée :
condition_NI_RN = (df["id_acteur"].isin(liste_NI_RN)) & (
    df["dateSeance_ts"].dt.normalize() < date_seuil
)  # dt.normalize() pour ignorer l'heure et éviter soucis de bornes


# Application de la modalité uniquement pour les lignes correspondant à la condition
df.loc[condition_NI_RN, "affiliation"] = "RN"

# Vérification
print("Lignes recodées RN :", condition_NI_RN.sum())
print(
    "Affiliation recodées pour",
    df.loc[condition_NI_RN, "id_acteur"].nunique(),
    "id_acteur uniques",
)

# vérification des cas sans affiliation :
print("Ceci ne modifie pas nombre sans affiliation : simple recodage NI vers RN")


Lignes recodées RN : 6701
Affiliation recodées pour 10 id_acteur uniques
Ceci ne modifie pas nombre sans affiliation : simple recodage NI vers RN


### Création d'une variable sur-imprimant l'appartenance au gouv

In [12]:
# ========== Création variable avec GOUV ==========

# renvoyer les membres du gouv à une catégorie "GOUV" pour les différencier
"""
nb : traçabilité
/!\ ici on veut récup membres du gouv, souvent en sans affiliation
mais on veut aussi forcer leur etiquette gvt même quand ils ont une affiliation de député
(ex : ministre qui est aussi député)

Logique de recodage :
Recoder membres GVT, uniquement si != PA0 (= garder cohérence avec cas précédents)
si une des conditions suivantes est vérifiée,
- ministre -> ok, 96 personnes pour 130 qualité, mais exclure le cas de Justin Trudeau et 19 cas PA0
- garde des sceaux (pas toujours co-qualifié de ministre) : ok, 2 bien Dupond-Moretti / Belloubet (même si 10 PA0)
- secrétaire d’État -> 40 personnes pour 53 qualité correspondantes, OK (2 PA0)
= basé sur la lecture des résultats de :
df["qualite_orateur"].value_counts()

-> mais il faut exclure "Premier ministre du Canada" -> 2 occurences 
Autre option : exclure des PA PA-107309 = Justin Trudeau, Premier ministre du Canada
"""

# masque condition membres gouvernement
mask_gvt = (
    df["qualite_orateur"].str.contains(
        "ministre|garde des sceaux|secrétaire d[’']État",
        case=False,
        na=False,
        regex=True,
    )
    & (df["id_acteur"] != "PA0")
    & (df["id_acteur"] != "PA-107309")
)  # exclure Justin Trudeau, "Premier ministre du Canada"

# ========== Création nouvelle variable avec GVT ==========
df["affiliation_et_gouv"] = df["affiliation"]  # conserver l'affiliation initiale
df.loc[mask_gvt, "affiliation_et_gouv"] = "GOUV"

# Vérification des cas affectés recodage GOUV
print("Lignes recodées GOUV :", mask_gvt.sum())
print(
    "Affiliation recodées pour",
    df.loc[mask_gvt, "id_acteur"].nunique(),
    "id_acteur uniques",
)

Lignes recodées GOUV : 110358
Affiliation recodées pour 109 id_acteur uniques


# GESTION DES CAS RESTANTS

In [13]:
# ======================
# TODO: AFFILIATIONS :
# ======================
# TODO : explorer les affiliation manquantes pour identifier les cas limites


In [14]:
# vérification des cas sans affiliation :
print(
    "Nombre restant d'interventions sans affiliation :",
    df["affiliation"].isna().sum(),
)
print(
    "Nombre restant d'id_acteur uniques sans affiliation :",
    df[df["affiliation"].isna()]["id_acteur"].nunique(),
)

print(
    "Nombre restant d'interventions sans affiliation_et_gouv :",
    df["affiliation_et_gouv"].isna().sum(),
)
print(
    "Nombre restant d'id_acteur uniques sans affiliation_et_gouv :",
    df[df["affiliation_et_gouv"].isna()]["id_acteur"].nunique(),
)

Nombre restant d'interventions sans affiliation : 55603
Nombre restant d'id_acteur uniques sans affiliation : 174
Nombre restant d'interventions sans affiliation_et_gouv : 12014
Nombre restant d'id_acteur uniques sans affiliation_et_gouv : 131


In [15]:
# IDENTIFICATION CAS MANQUANTS ET LIMITES AFFILIATION ET GOUV

# Acteurs avec au moins un NA dans affiliation_et_gouv (hors PA0)
restant_affiliation_et_gouv = df[
    (df["affiliation_et_gouv"].isna()) & (df["id_acteur"] != "PA0")
]

# Comptage NA par acteur directement
count_restant_par_acteur = (
    restant_affiliation_et_gouv.groupby(
        ["id_acteur", "nom_orateur_clean"], dropna=False
    )
    .size()
    .reset_index(name="nb_na_interventions")
)

# Répartition NA / renseigné pour ces mêmes acteurs
repartition = (
    df[df["id_acteur"].isin(count_restant_par_acteur["id_acteur"])]
    .groupby("id_acteur")["affiliation_et_gouv"]
    .agg(
        nb_na=lambda s: s.isna().sum(),
        nb_renseigne=lambda s: s.notna().sum(),
    )
    .reset_index()
)

resultat = count_restant_par_acteur.merge(repartition, on="id_acteur").sort_values(
    "nb_renseigne", ascending=False
)

print(f"Nombre d'id_acteur avec au moins un NA : {resultat['id_acteur'].nunique()}")
display(resultat)

resultat.to_csv("../data/temp/count_restant_affiliation_et_gouv.csv", index=False)


Nombre d'id_acteur avec au moins un NA : 130


,id_acteur,nom_orateur_clean,nb_na_interventions,nb_na,nb_renseigne
129,PA773443,M. Éric Dupond-Moretti,6,6,6793
126,PA702191,Mme Nicole Belloubet,4,4,3644
128,PA729332,M. Julien Denormandie,3,3,2258
127,PA717159,Mme Frédérique Vidal,1,1,1103
125,PA205600,Mme Florence Parly,1,1,515
...,...,...,...,...,...
38,PA-121499,M. Guillaume Gauthier,8,8,0
37,PA-121489,M. Philippe Chalmin,10,10,0
36,PA-121449,M. Pierre Moscovici,59,59,0
35,PA-121439,Mme Sophie Moati,7,7,0


In [16]:
# IDENTIFICATION CAS MANQUANTS ET LIMITES AFFILIATION

# Acteurs avec au moins un NA dans affiliation (hors PA0)
restant_affiliation = df[(df["affiliation"].isna()) & (df["id_acteur"] != "PA0")]

# Comptage NA par acteur directement
count_restant_par_acteur = (
    restant_affiliation.groupby(["id_acteur", "nom_orateur_clean"], dropna=False)
    .size()
    .reset_index(name="nb_na_interventions")
)

# Répartition NA / renseigné pour ces mêmes acteurs
repartition = (
    df[df["id_acteur"].isin(count_restant_par_acteur["id_acteur"])]
    .groupby("id_acteur")["affiliation"]
    .agg(
        nb_na=lambda s: s.isna().sum(),
        nb_renseigne=lambda s: s.notna().sum(),
    )
    .reset_index()
)

resultat_affiliation = count_restant_par_acteur.merge(
    repartition, on="id_acteur"
).sort_values("nb_renseigne", ascending=False)

print(
    f"Nombre d'id_acteur avec au moins un NA : {resultat_affiliation['id_acteur'].nunique()}"
)
display(resultat_affiliation)

resultat_affiliation.to_csv("../data/temp/count_restant_affiliation.csv", index=False)

Nombre d'id_acteur avec au moins un NA : 173


,id_acteur,nom_orateur_clean,nb_na_interventions,nb_na,nb_renseigne
0,PA-100799,M. Bruno Retailleau,2,2,0
119,PA-125769,M. Patrick Weil,8,8,0
111,PA-125689,M. Grégoire Lefebure,2,2,0
112,PA-125699,Mme Marine Malberg,4,4,0
113,PA-125709,M. Thierry Malbert,5,5,0
...,...,...,...,...,...
59,PA-124629,Mme Laurence Rossignol,1,1,0
60,PA-124769,Mme Maryse Carrère,1,1,0
61,PA-124779,Mme Mélanie Vogel,1,1,0
62,PA-125019,Mme Sophie Taillé-Polian,9,9,0


##### LES SOUCIS POSSIBLES multi affil:


In [17]:
# Cas où un même id_acteur a plusieurs valeurs différentes de affiliation_et_gouv
tmp = df[["id_acteur", "nom_orateur_clean", "affiliation_et_gouv"]].copy()
tmp["affiliation_et_gouv_norm"] = tmp["affiliation_et_gouv"].fillna("<<NA>>")

# ids avec au moins 2 modalités différentes (en comptant NA)
ids_multi_affil = (
    tmp.groupby("id_acteur")["affiliation_et_gouv_norm"]
    .nunique()
    .loc[lambda s: s > 1]
    .index
)
cas_diff = tmp[tmp["id_acteur"].isin(ids_multi_affil)].copy()

print(
    "Nombre d'id_acteur avec plusieurs valeurs de affiliation_et_gouv :",
    len(ids_multi_affil),
)
print("Nombre total de lignes concernées :", len(cas_diff))

cas_multi_affil = (
    cas_diff.groupby(["id_acteur", "nom_orateur_clean"])["affiliation_et_gouv_norm"]
    .agg(lambda x: sorted(set(x)))
    .reset_index(name="valeurs_affiliation_et_gouv")
    .sort_values(["nom_orateur_clean", "id_acteur"])
)
display(cas_multi_affil)
cas_multi_affil.to_csv("../data/temp/cas_multi_affiliation_et_gouv.csv", index=False)

Nombre d'id_acteur avec plusieurs valeurs de affiliation_et_gouv : 162
Nombre total de lignes concernées : 176008


,id_acteur,nom_orateur_clean,valeurs_affiliation_et_gouv
93,PA720422,M. Adrien Quatennens,"[LFI, NI]"
137,PA722086,M. Adrien Taquet,"[GOUV, NI, REN]"
156,PA794914,M. Alexandre Vincendet,"[HOR, LR]"
27,PA421348,M. André Villiers,"[HOR, UDI]"
8,PA267355,M. Antoine Herth,"[AGIR-E, UDI]"
...,...,...,...
22,PA336175,Mme Sylvia Pinel,"[LIOT, NI]"
95,PA720500,Mme Valérie Petit,"[AGIR-E, REN]"
63,PA719194,Mme Yolaine de Courson,"[DEM, EDS, NI, REN]"
50,PA717161,Mme Élisabeth Borne,"[GOUV, REN]"


##### LES SOUCIS POSSIBLES AVEC MEMBRES GOUV :
- parfois pas d'info
- mais pas bien identifié gouv (rapporteur, infon manquante, etc.)
- actuellement, comme avant ça on force l'affil, on s'en rend pas compte
- les vérifier

In [18]:
# cf FAIRE plus restrictif ? = depuis l'affil députés, pour éviter de rater des cas
# ou on aurait forcé un groupe à un ministre et qu'on flague pas qu'il est pas identifié comme ministre sur une intervention où il l'est

# TODO: comparer ce que ça donne pour gouv entre affiliation_mandat_députés et affiliation_et_gouv pour voir si ça correspond bien
# = cf on en trouvait sans doute comme ça des vides dans affiliation_mandat_députés qui étaient en fait des membres du gouv
# et que là on recode selon leur affiliation et pas en GOUV si l'info qualité orateur est pas bonne
# pas identifié ministre machin car info manquante, autre statut comme rapporteur, etc.

# TODO: géréer les cas limites gouv quand sont rapporteurs, etc. (darmanin, EDM, etc.)
# TODO : sans doute donc gérer chaque type qui a GVT et autre affilaition pour voir si normal ?

In [19]:
# CAS LIMITE GOUV :
# ORATEURS AVEC AFFIL = GOUV + AUTRE CHOSE : identification + comptage des interventions

# LOGIQUE :
# - RENVOYER LES CAS OU AFFIL = GOUV + AUTRE CHOSE
# - identification orateurs et combinaison d'affiliations
# - comptage des interventions

# Interventions des orateurs qui ont plusieurs affiliations dont GVT

# 1) Ne garder que les lignes avec affiliation renseignée
temp = df.loc[
    df["affiliation_et_gouv"].notna(),
    ["id_acteur", "nom_orateur_clean", "affiliation_et_gouv"],
].copy()
# (ça ici permet les cas sans affil forcée, mais marcherait aussi si on fait
# juste après l'affil dynamique pour pas rater les cas)
temp["affiliation_et_gouv_norm"] = temp["affiliation_et_gouv"].fillna("<<NA>>")

# 2) Profils d'affiliation par id_acteur → garder ceux avec affiliations multiples dont GOUV
affil_par_id = temp.groupby("id_acteur")["affiliation_et_gouv_norm"].agg(
    lambda s: sorted(set(s.astype(str)))
)
ids_multi_avec_gvt = affil_par_id[
    affil_par_id.apply(lambda x: len(x) > 1 and "GOUV" in x)
].index

# 3) Comptage GOUV vs AUTRE par orateur
subset = temp[temp["id_acteur"].isin(ids_multi_avec_gvt)].copy()
subset["type_intervention"] = (
    subset["affiliation_et_gouv"].eq("GOUV").map({True: "GOUV", False: "AUTRE"})
)

counts = subset["type_intervention"].value_counts()
nb_gouv = int(counts.get("GOUV", 0))
nb_autre = int(counts.get("AUTRE", 0))
print(f"Interventions comme membre du gouv : {nb_gouv}")
print(f"Interventions dans les autres cas  : {nb_autre}")
print(f"Total                              : {nb_gouv + nb_autre}")

# 4) Tableau fusionné : comptages + affiliations
par_orateur = (
    subset.groupby(["id_acteur", "nom_orateur_clean", "type_intervention"])
    .size()
    .unstack(fill_value=0)
    .reset_index()
)
par_orateur["TOTAL"] = par_orateur.get("GOUV", 0) + par_orateur.get("AUTRE", 0)

affiliations = (
    subset.groupby(["id_acteur", "nom_orateur_clean"])["affiliation_et_gouv_norm"]
    .agg(lambda s: sorted(set(s.astype(str))))
    .reset_index(name="affiliations")
)

membres_gouv_multi_affil = par_orateur.merge(
    affiliations, on=["id_acteur", "nom_orateur_clean"]
).sort_values("TOTAL", ascending=False)
print(
    f"\nNombre d'orateurs avec affiliations multiples dont GOUV : {len(membres_gouv_multi_affil)}"
)
display(membres_gouv_multi_affil)

membres_gouv_multi_affil.to_csv(
    "../data/temp/membres_gouv_multi_affil.csv", index=False
)


Interventions comme membre du gouv : 59580
Interventions dans les autres cas  : 31818
Total                              : 91398

Nombre d'orateurs avec affiliations multiples dont GOUV : 52


,id_acteur,nom_orateur_clean,AUTRE,GOUV,TOTAL,affiliations
9,PA607846,M. Gérald Darmanin,17,8435,8452,"[GOUV, REN]"
2,PA330357,M. Olivier Dussopt,154,6016,6170,"[GOUV, REN, SOC-A]"
12,PA642788,M. Olivier Véran,2105,3744,5849,"[GOUV, REN]"
0,PA267336,M. Joël Giraud,4503,192,4695,"[GOUV, REN]"
3,PA331481,M. Bruno Le Maire,1,4479,4480,"[GOUV, REN]"
15,PA717161,Mme Élisabeth Borne,1,4302,4303,"[GOUV, REN]"
44,PA759832,Mme Agnès Pannier-Runacher,2,4041,4043,"[GOUV, REN]"
42,PA722190,M. Gabriel Attal,271,3348,3619,"[GOUV, REN]"
4,PA331582,M. Philippe Vigier,3264,190,3454,"[DEM, GOUV, LIOT, UDI]"
31,PA721134,M. Roland Lescure,1503,1941,3444,"[GOUV, REN]"


In [20]:
# ==========================================================
# CAS LIMITES GOUV :
# valeurs manquantes d'affiliation_et_gouv "encadrées" par GOUV
# ==========================================================

# Colonnes utiles pour audit
cols_audit = [
    "id_acteur",
    "nom_orateur_clean",
    "dateSeance_ts",
    "affiliation_et_gouv",
    "affiliation",
    "affiliation_mandat_députés",
    "qualite_orateur",
    "code_parole",
    "id_syceron",
    "texte"
]

tmp = df.loc[
    df["id_acteur"].notna() & (df["id_acteur"] != "PA0"),
    cols_audit,
].copy()

tmp = tmp.sort_values(["id_acteur", "dateSeance_ts"]).reset_index(drop=True)
g = tmp.groupby("id_acteur", group_keys=False)

# Valeurs non manquantes les plus proches avant/après
tmp["prev_non_na_affil"] = g["affiliation_et_gouv"].ffill()
tmp["next_non_na_affil"] = g["affiliation_et_gouv"].bfill()

# Dates de référence (où affiliation_et_gouv est connue)
tmp["date_affil_connue"] = tmp["dateSeance_ts"].where(
    tmp["affiliation_et_gouv"].notna()
)
tmp["date_connue_avant"] = g["date_affil_connue"].ffill()
tmp["date_connue_apres"] = g["date_affil_connue"].bfill()

# NA strictement entre deux observations GOUV
mask_na_entre_gouv = (
    tmp["affiliation_et_gouv"].isna()
    & (tmp["prev_non_na_affil"] == "GOUV")
    & (tmp["next_non_na_affil"] == "GOUV")
)

cas_na_entre_gouv = tmp.loc[
    mask_na_entre_gouv,
    [
        "id_acteur",
        "nom_orateur_clean",
        "dateSeance_ts",
        "date_connue_avant",
        "date_connue_apres",
        "qualite_orateur",
        "code_parole",
        "id_syceron",
        "affiliation_et_gouv",
        "affiliation",
        "affiliation_mandat_députés",
        "texte"
    ],
].copy()

print("Interventions NA entre deux bornes GOUV :", len(cas_na_entre_gouv))
print("id_acteur uniques concernés :", cas_na_entre_gouv["id_acteur"].nunique())
display(cas_na_entre_gouv.head(20))

# --- Regrouper en "segments" de NA consécutifs par acteur ---
tmp["is_gap_gouv"] = mask_na_entre_gouv
tmp["gap_start"] = tmp["is_gap_gouv"] & ~g["is_gap_gouv"].shift(fill_value=False)
tmp["gap_num"] = g["gap_start"].cumsum()
tmp.loc[~tmp["is_gap_gouv"], "gap_num"] = pd.NA

segments_na_entre_gouv = (
    tmp.loc[tmp["is_gap_gouv"]]
    .groupby(["id_acteur", "nom_orateur_clean", "gap_num"], dropna=False)
    .agg(
        date_debut_na=("dateSeance_ts", "min"),
        date_fin_na=("dateSeance_ts", "max"),
        nb_interventions_na=("dateSeance_ts", "size"),
        borne_gouv_avant=("date_connue_avant", "first"),
        borne_gouv_apres=("date_connue_apres", "first"),
        nb_qualite_gouv_explicite=(
            "qualite_orateur",
            lambda s: s.astype(str)
            .str.contains(
                "ministre|garde des sceaux|secrétaire d[’']État",
                case=False,
                na=False,
                regex=True,
            )
            .sum(),
        ),
    )
    .reset_index()
    .sort_values(["nb_interventions_na", "date_debut_na"], ascending=[False, True])
)

print("Segments NA entre bornes GOUV :", len(segments_na_entre_gouv))
display(segments_na_entre_gouv.head(30))

# Exports pour revue manuelle
cas_na_entre_gouv.to_csv("../data/temp/cas_na_entre_gouv_lignes.csv", index=False)
segments_na_entre_gouv.to_csv("../data/temp/cas_na_entre_gouv_segments.csv", index=False)

Interventions NA entre deux bornes GOUV : 15
id_acteur uniques concernés : 5


,id_acteur,nom_orateur_clean,dateSeance_ts,date_connue_avant,date_connue_apres,qualite_orateur,code_parole,id_syceron,affiliation_et_gouv,affiliation,affiliation_mandat_députés,texte
19332,PA205600,Mme Florence Parly,2019-05-28 15:00:00,2019-05-28 15:00:00,2019-05-28 15:00:00,NaN,PAROLE_1_2,1736302,NaN,NaN,None,Quand bien même ce serait le cas – ce qui n’es...
231970,PA702191,Mme Nicole Belloubet,2024-04-02 15:00:00,2024-04-02 15:00:00,2024-04-08 21:45:00,NaN,non_précisé,3421170,NaN,NaN,None,Nous avons affirmé un objectif clair : d’une p...
231971,PA702191,Mme Nicole Belloubet,2024-04-02 15:00:00,2024-04-02 15:00:00,2024-04-08 21:45:00,NaN,non_précisé,3421173,NaN,NaN,None,"Toutefois, des brassages seront possibles entr..."
231972,PA702191,Mme Nicole Belloubet,2024-04-02 15:00:00,2024-04-02 15:00:00,2024-04-08 21:45:00,NaN,non_précisé,3421177,NaN,NaN,None,"Dans la même période, nous avons créé 12 000 e..."
231973,PA702191,Mme Nicole Belloubet,2024-04-02 15:00:00,2024-04-02 15:00:00,2024-04-08 21:45:00,NaN,non_précisé,3421179,NaN,NaN,None,"C’est dire l’effort qui a été fait.Enfin, dans..."
241767,PA717159,Mme Frédérique Vidal,2020-09-22 15:00:00,2020-09-22 15:00:00,2020-09-22 15:00:00,NaN,non_précisé,2189198,NaN,NaN,None,Nous parlons de la recherche !
554434,PA729332,M. Julien Denormandie,2020-07-16 15:00:00,2020-07-16 15:00:00,2020-07-16 15:00:00,NaN,PAROLE_1_2,2156328,NaN,NaN,None,"Oui, mon ami Stéphane Travert, que je salue. L..."
554436,PA729332,M. Julien Denormandie,2020-07-16 15:00:00,2020-07-16 15:00:00,2020-07-16 15:00:00,NaN,PAROLE_1_2,2156362,NaN,NaN,None,"L’après-midi même, je me suis rendu dans une e..."
554535,PA729332,M. Julien Denormandie,2020-10-05 16:00:00,2020-10-05 16:00:00,2020-10-05 16:00:00,NaN,non_précisé,2210321,NaN,NaN,None,Exactement !
566643,PA773443,M. Éric Dupond-Moretti,2020-07-30 15:00:00,2020-07-30 15:00:00,2020-07-30 15:00:00,NaN,non_précisé,2170932,NaN,NaN,None,Souriez !


Segments NA entre bornes GOUV : 12


,id_acteur,nom_orateur_clean,gap_num,date_debut_na,date_fin_na,nb_interventions_na,borne_gouv_avant,borne_gouv_apres,nb_qualite_gouv_explicite
1,PA702191,Mme Nicole Belloubet,1.0,2024-04-02 15:00:00,2024-04-02 15:00:00,4,2024-04-02 15:00:00,2024-04-08 21:45:00,0
0,PA205600,Mme Florence Parly,1.0,2019-05-28 15:00:00,2019-05-28 15:00:00,1,2019-05-28 15:00:00,2019-05-28 15:00:00,0
3,PA729332,M. Julien Denormandie,1.0,2020-07-16 15:00:00,2020-07-16 15:00:00,1,2020-07-16 15:00:00,2020-07-16 15:00:00,0
4,PA729332,M. Julien Denormandie,2.0,2020-07-16 15:00:00,2020-07-16 15:00:00,1,2020-07-16 15:00:00,2020-07-16 15:00:00,0
6,PA773443,M. Éric Dupond-Moretti,1.0,2020-07-30 15:00:00,2020-07-30 15:00:00,1,2020-07-30 15:00:00,2020-07-30 15:00:00,0
7,PA773443,M. Éric Dupond-Moretti,2.0,2020-09-17 15:00:00,2020-09-17 15:00:00,1,2020-09-17 15:00:00,2020-09-17 15:00:00,0
2,PA717159,Mme Frédérique Vidal,1.0,2020-09-22 15:00:00,2020-09-22 15:00:00,1,2020-09-22 15:00:00,2020-09-22 15:00:00,0
5,PA729332,M. Julien Denormandie,3.0,2020-10-05 16:00:00,2020-10-05 16:00:00,1,2020-10-05 16:00:00,2020-10-05 16:00:00,0
8,PA773443,M. Éric Dupond-Moretti,3.0,2020-10-20 15:00:00,2020-10-20 15:00:00,1,2020-10-20 15:00:00,2020-10-20 15:00:00,0
9,PA773443,M. Éric Dupond-Moretti,4.0,2023-07-03 16:00:00,2023-07-03 16:00:00,1,2023-07-03 16:00:00,2023-07-03 16:00:00,0


In [21]:
# ==============================
# vérif et possibles soucis :
# ==============================

# TODO: matthias & Léo : check les cas particuliers.

# TODO : voir avec matthias si renvoi derniere affiliation suffit pour couleur politique globale.
# ou si on affine pour membre gouv qui étaient député y a longtemps
# (genre ici nous bachelot serait UMP -> recod LR
# mais donc discutable et voir gestion manuelle membres gouv ?)

# TODO : cas limites breneel, boyer -> hors bornes, mais possible qu'ils reviennent comme intervenants externes en fait ?

# TODO: autre option enchainement
# 1 tempo dynamique
# 2 var affil_gouv : en ajoutant les cas qualite_orateur
# 3 identifier les cas limites quand gouv + info manquante parfois
# 4 si il faut les ajouter en "doute"
# 5 forcer les affil restantes par dessus avec groupeAbrev.
# 6 et au pire donc pour la var affil complète, réimposer les groupes pour le gouv


### Trucs Matthias
Mais possiblement ok depuis qu'on force l'allifilation ? 

In [22]:
# TODO : voir si jamais c'est justifié (reviendrait pas comme député ou gouv mais membre externe)

# # solution temporaire sur 2 cas étranges
# LM = OK QUAND ON FORCE LES AFFIL + pas identifié à cause bornes


# Boyer = ["PA330684"]  # cas similaire sur intervention du 7 novembre 2020
# LM explication : l'info d'affiliation s'arrête au 30 sept 2020
# "PM731296";"PA330684";"15";"2017-06-27";"2020-09-30";"20";"1";"Membre";"PO730934";"Les Républicains";"LR";"LR";"2017-06-27";"2022-06-21"

# df.loc[df["id_acteur"].isin(Boyer), "groupe&gvt_affiliation"] = "LR"

# Bruneel = [
#     "PA720546"
# ]  # ici cas étrange sur une intervention le 9 janvier 2023, il a été laissé en valeur manquante alors que GDR
# LM = l'info affiliation s'arrête au 21 juin 2022
# "PM731533";"PA720546";"15";"2017-06-27";"2022-06-21";"20";"1";"Membre";"PO730940";"Gauche démocrate et républicaine";"GDR";"GDR";"2017-06-27";"2022-06-21"

# df.loc[df["id_acteur"].isin(Bruneel), "groupe&gvt_affiliation"] = "GDR"

In [23]:
# TODO : on aura sans doute décidé plus haut, je garde au cas où pour l'instant
# TODO : léo, voir ces machins avec matthias ensuite pour clarifier.
# Et voir pourquoi passé par un isin plutôt que ==

# # Reste des cas particuliers à replacer dans leur affiliation au moment de leurs fonctions gouvernementales respectives
# Bachelot = ["PA332"]

# df.loc[df["id_acteur"].isin(Bachelot), "groupe_all_affiliation"] = (
#     "NI"  # NI ou mettre valeur manquante ? pareil pour Philippe, Le Drian, Rousseau
# )

# Vautrin = ["PA267797"]

# df.loc[df["id_acteur"].isin(Vautrin), "groupe_all_affiliation"] = "REN"

# Philippe = ["PA345619"]

# df.loc[df["id_acteur"].isin(Philippe), "groupe_all_affiliation"] = "NI"

# Ledrian = ["PA1872"]

# df.loc[df["id_acteur"].isin(Ledrian), "groupe_all_affiliation"] = "NI"

# Rousseau = ["PA826635"]

# df.loc[df["id_acteur"].isin(Rousseau), "groupe_all_affiliation"] = "NI"

## Export

In [24]:
# Export du csv nettoyé
df.to_csv("../data/interim/data_cleaning_full.csv", index=False)

# # NB: certaines col du df_deputes introduisent une erreur à l'import/export
# # Elles ne sont pas utilisées ici, mais si besoin de les utiliser
# # forcer le QUOTE_ALL permet de résoudre
# # (cf : adresses et réseaux sociaux contenant saut de lignes = erreurs de parsing (cas eric.martineau))

# PROVISOIRE !! Regrouper les interventions interrompues
ÇA NE MARCHE PAS POUR L'INSTANT !!!!!!


In [25]:
# TODO: À affiner et vérifier la fusion interventions interrompues
# FIXME: ça foire

# AVISER : pas le cas ici, mais envisager possible gestion des cas NaN
df_interruption = df[df["code_grammaire"].str.contains("INTERRUPTION")]
df_intervention = df[~df["code_grammaire"].str.contains("INTERRUPTION")]
# Si il fallait s'en assurer :
# is_interruption = df["code_grammaire"].str.contains("INTERRUPTION", na=False)
# df_interruption = df[is_interruption]
# df_intervention = df[~is_interruption]

# assert len(df_interruption) + len(df_intervention) == len(df), (
#     f"Lignes perdues lors du split ! "
#     f"{len(df)} ≠ {len(df_interruption)} + {len(df_intervention)} "
#     f"(NaN dans code_grammaire : {df['code_grammaire'].isna().sum()})"
# )

# TODO : NON ÇA VA PAS ÇA REGROUPE NAWAK ????

# ordinal_prise semble plus précis au niveau des intervenants
# = est constant quand interrompu là où les ordres obsolu et ptsodj changent
group_keys = ["uid", "dateSeance_ts", "id_acteur", "ordinal_prise"]

# agréger : concat texte, sommer longueur, garder premières infos utiles
agg = {
    "texte": lambda s: " ".join(s.dropna().astype(str)).strip(),
    "len_texte_brut": "sum",
    "code_parole": lambda s: ", ".join(sorted(set(s.dropna().astype(str)))),
    "id_syceron": lambda s: s.dropna().unique().tolist(),
    # "ordre_absolu_seance": "first", # list pour garder l'ordre des prises ?
    # "nom_orateur": "first",
    # "qualite_orateur": "first",
    # "id_orateur": "first",
    # "stime": "first",
}

# ajouter 'first' pour toutes les autres colonnes non clés/non déjà agrégées
for c in df_intervention.columns:
    if c not in group_keys and c not in agg:
        agg[c] = "first"

# Regroupe les interventions par clés communes et agrège les colonnes définies dans `agg`
df_intervention_grouped = (
    df_intervention.groupby(group_keys, dropna=False).agg(agg).reset_index()
)

# Recolle les interventions regroupées avec les interruptions
# puis aligne les colonnes sur le format d'origine
df_concat = pd.concat([df_intervention_grouped, df_interruption], ignore_index=True)[
    df_interruption.columns
]

# Retrier dans l'ordre chronologique et d'affichage de la séance
# nb : ici ok car gardé seulement first pour ordre_absolu_seance
# mais modif si jamais on avait gardé la liste complète des ordres
df_concat = df_concat.sort_values(
    by=["dateSeance_ts", "valeur_ptsodj", "ordre_absolu_seance"]
).reset_index(drop=True)

print(
    f"Regroupement des interventions interrompues \n"
    f"avant: {len(df)} | après: {len(df_concat)} "
    f"(interventions: de {len(df_intervention)} → à {len(df_intervention_grouped)}, "
    f"interruptions: {len(df_interruption)})"
)

# TODO : len_texte_brut sera à revoir pour interv groupée
# , car là on à la trace de la longueur des interventions avant regroupement
# puis on en fait une somme      "len_texte_brut": "sum",
# donc voir si clair pour une fois regroupé ?, ou si on recalcule plutôt une nouvelle var ?
# ie si peut porter à confusion

Regroupement des interventions interrompues 
avant: 683680 | après: 425561 (interventions: de 450963 → à 192844, interruptions: 232717)


In [26]:
# Export du csv concat nettoyé
df_concat.to_csv("../data/interim/data_cleaning_grouped.csv", index=False)

# # NB: certaines col du df_deputes introduisent une erreur à l'import/export
# # Elles ne sont pas utilisées ici, mais si besoin de les utiliser
# # forcer le QUOTE_ALL permet de résoudre
# # (adresses et réseaux sociaux contenant saut de lignes = erreurs de parsing (cas eric.martineau))

# EXPLORATION

In [27]:
# TODO : aller voir parce que ça regroupe quand meme des trucs qui
# ont pas le même code parole
# donc voir le pourquoi du comment
# MAIS ON S'EN COGNE UN PEU SUR LE PRINCIPE ?
# ENFIN AVISER QUE JUSTE LES AVIS GOUV SOIT PAS REGROUPÉS
# AVEC UNE PRISE PAROLE PLUS LARGE ?
# CHANGE RIEN DE DRAMATIQUE SANS DOUTE.

df_concat["code_parole"].value_counts()[:-10]

code_parole
non_précisé                                 287786
PAROLE_1_2                                   92122
PAROLE_1_2, non_précisé                      15040
AVIS_COM_1_20                                10824
AVIS_GVT_1_20                                10097
AVIS_GVT_1_20, PAROLE_1_2                     3371
AVIS_COM_1_20, non_précisé                    2265
AVIS_COM_1_20, PAROLE_1_2                     1787
AVIS_GVT_1_20, non_précisé                    1096
AVIS_COM_1_20, PAROLE_1_2, non_précisé         684
AVIS_GVT_1_20, PAROLE_1_2, non_précisé         453
AVIS_COM_1_20, AVIS_GVT_1_20                    14
AVIS_COM_1_20, AVIS_GVT_1_20, PAROLE_1_2         5
Name: count, dtype: int64

In [28]:
# ÇA REGROUPE NAWAK !!!!!

In [29]:
chelou = df_concat[df_concat["code_parole"] == "AVIS_GVT_1_20, PAROLE_1_2"]
chelou.head

<bound method NDFrame.head of                           uid               SeanceRef   SessionRef  \
430     CRSANR5L15S2017E1N003                    None         None   
467     CRSANR5L15S2017E1N004                    None         None   
817     CRSANR5L15S2017E1N006                    None         None   
897     CRSANR5L15S2017E1N007                    None         None   
1068    CRSANR5L15S2017E1N008                    None         None   
...                       ...                     ...          ...   
406306  CRSANR5L16S2024O1N150  RUANR5L16S2024IDS28146  SCR5A2024O1   
406492  CRSANR5L16S2024O1N151  RUANR5L16S2024IDS28218  SCR5A2024O1   
406525  CRSANR5L16S2024O1N151  RUANR5L16S2024IDS28218  SCR5A2024O1   
406594  CRSANR5L16S2024O1N152  RUANR5L16S2024IDS28157  SCR5A2024O1   
410411  CRSANR5L16S2024O1N170  RUANR5L16S2024IDS28206  SCR5A2024O1   

               dateSeance         dateSeanceJour numSeanceJour  numSeance  \
430     20170706093000000  jeudi 06 juillet 2017    

Je comprends bien l’intention exposée par M. Lagarde. Je rappelle toutefois l’existence de la disposition dont vient de parler M. le rapporteur.De surcroît, le contrôle des assemblées a été sensiblement renforcé lors de la quatrième prorogation de l’état d’urgence, en juillet 2016. Dès le 25 juillet 2016, les commissions des lois des deux assemblées se sont ainsi vu transmettre copie des mesures prises sur le fondement de la loi du 3 avril 1955, ce qui a permis aux rapporteurs concernés de disposer d’une connaissance exhaustive de toutes ces mesures. J’en ai parlé abondamment au Sénat, hier, avec M. le rapporteur Michel Mercier. Les rapporteurs des deux commissions des lois sont informés de tout ce qui se passe pendant l’état d’urgence. L’avis du Gouvernement est donc défavorable. Vous me permettrez d’exprimer mon accord avec M. Larrivé : les moyens de contrôle dont disposent aujourd’hui les commissions des lois de l’Assemblée nationale et du Sénat sont très importants. Les informations les plus confidentielles sont communiquées à leurs présidents et rapporteurs respectifs. Vous comprendrez aisément que, pendant la Guerre de Quatorze, le comité parlementaire était informé des grandes lignes stratégiques, pas forcément de la tactique déployée sur le terrain. L’avis du Gouvernement reste donc défavorable.

Je comprends bien l’intention exposée par M. Lagarde. Je rappelle toutefois l’existence de la disposition dont vient de parler M. le rapporteur.De surcroît, le contrôle des assemblées a été sensiblement renforcé lors de la quatrième prorogation de l’état d’urgence, en juillet 2016. Dès le 25 juillet 2016, les commissions des lois des deux assemblées se sont ainsi vu transmettre copie des mesures prises sur le fondement de la loi du 3 avril 1955, ce qui a permis aux rapporteurs concernés de disposer d’une connaissance exhaustive de toutes ces mesures. J’en ai parlé abondamment au Sénat, hier, avec M. le rapporteur Michel Mercier. Les rapporteurs des deux commissions des lois sont informés de tout ce qui se passe pendant l’état d’urgence. L’avis du Gouvernement est donc défavorable.

In [30]:
df[df["id_syceron"] == 983326][["nom_orateur_clean", "texte", "len_texte_brut"]]

,nom_orateur_clean,texte,len_texte_brut
78246,M. Gérard Collomb,Je comprends bien l’intention exposée par M. L...,791.0


In [31]:
df[df["id_syceron"] == 983358]

,uid,SeanceRef,SessionRef,dateSeance,dateSeanceJour,numSeanceJour,numSeance,typeAssemblee,legislature,session,...,nombreMandats,experienceDepute,scoreParticipation,scoreLoyaute,scoreMajorite,dateMaj,dateSeance_ts,affiliation_mandat_députés,affiliation,affiliation_et_gouv
78261,CRSANR5L15S2017E1N003,NaN,NaN,20170706093000000,jeudi 06 juillet 2017,1,3,AN,15,Première session extraordinaire 2017,...,NaN,NaN,NaN,NaN,NaN,NaN,2017-07-06 09:30:00,None,NaN,GOUV


In [32]:
# Vérifier si code_parole varie au sein d'un même groupe
check = df_intervention.groupby(group_keys, dropna=False)["code_parole"].nunique()
print("Groupes avec code_parole non constant :", (check > 1).sum())

Groupes avec code_parole non constant : 24724
